### DIGI405 Text Analysis Project Notebook

[0.2.5 - 2025-10-08](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - optional functionality to run inference on augmented data or data from a CSV  
[0.2.3 - 2025-09-15](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - metric ordering    
[0.2.2 - 2025-09-08](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - quality of life improvements, probability labels    
[0.2.1 - 2025-08-19](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - ensure filtering / grid search handled as expected   
Note: Search notebook for 0.2.1/0.2.2/0.2.3/0.2.5 to find changes if you want to apply to your existing notebook.

## Introduction

You should use this notebook as a starting point for your DIGI405 project. It provides code to select your dataset, and run a complete text classification pipeline with [textplumber](https://geoffford.nz/textplumber/), a package that provides an easy to use interface to methods covered in this course.

**Name:** Xian Zhou  
**Student ID:**  39111583
**Project option:** Sentiment  
**Project submission date:**  05/27/2026

Please also add your name to your notebook filename (where it says 'NAME').

### Notebook structure

Sections 1-4 provide code you should modify or extend. In your report, you can refer to code sections by their section number, eg 2.1.

## 1. Setup

You must select the Python 3.12 kernel to run the code in this notebook. 

In [30]:
from datasets import load_dataset, ClassLabel, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import confusion_matrix, classification_report

from textplumber.core import *
from textplumber.clean import *
from textplumber.preprocess import *
from textplumber.tokens import *
from textplumber.pos import *
from textplumber.embeddings import *
from textplumber.report import *
from textplumber.store import *
from textplumber.lexicons import *
from textplumber.textstats import *

from imblearn.under_sampling import RandomUnderSampler 

import warnings

# in the interests of readability, ignoring this warning
warnings.filterwarnings("ignore", message="Your stop_words may be inconsistent with your preprocessing")

These settings control the display of Pandas dataframes in the notebook.

In [31]:
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.max_colwidth', 500) # increase this to see more text in the dataframe

Get word lists: 
* The stop word list is from NLTK.   
* All of the word lists (including the stop word list) can be used to extract lexicon count features to extract features based on a set of words.

In [32]:
stop_words = get_stop_words()
stop_words_lexicon = {'stop_words': stop_words}
empath_lexicons = get_empath_lexicons()
vader_lexicons = get_sentiment_lexicons()

## 2. Load and inspect data

### 2.1 Choose a dataset and preview the labels

Below you can select a dataset for the assignment. The options are `sentiment`, `essay` and `genre`. Change the value of `dataset_option` below. The datasets available on Huggingface.co will be downloaded automatically and a link provided to the dataset card with more information. The `genre` dataset was distributed with this notebook.   

Note:  The `movie_reviews` dataset is being used to demonstrate the notebook and is not one of your options for the assignment.  

In [33]:
# Choose 'essay', 'sentiment', or 'genre' ('movie_reviews' is just for testing/demonstration)
#dataset_option = 'movie_reviews' 
dataset_option = 'sentiment' 
if dataset_option == 'movie_reviews':
	dataset_name = 'polsci/sentiment-polarity-dataset-v2.0'
	dataset_dir = None
	target_labels = ['neg', 'pos']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'train'
	print('The movie_reviews is to demonstrate the notebook and is not an assignment option.')
elif dataset_option == 'sentiment':
	dataset_name = 'cardiffnlp/tweet_eval'
	dataset_dir = 'sentiment'
	target_labels = ['negative', 'neutral', 'positive']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'validation'
	print('You selected the sentiment dataset. Read more about this at https://huggingface.co/datasets/cardiffnlp/tweet_eval')
elif dataset_option == 'essay':
	dataset_name = 'polsci/ghostbuster-essay-cleaned'
	dataset_dir = None
	target_labels = ['claude', 'gpt', 'human']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'test'
	print('You selected the essay dataset. Read more about this at https://huggingface.co/datasets/polsci/ghostbuster-essay-cleaned')
elif dataset_option == 'genre':
    dataset_name = 'genre'
    dataset_type = 'json'
	# Note: Quality of life improvement for version 0.2.2
    dataset_dir = '/srv/source-data/genre_dataset.json' # if you are running this locally change to the path on your machine
    target_labels = ['Fiction', 'Letter', 'Notice', 'Obituary', 'Poetry or verse', 'Recipe', 'Review']
    text_column = 'text'
    label_column = 'label'
    train_split_name = 'train'
    test_split_name = 'test'
    print('You selected the genre dataset.')
else:
	print('Try again! That was not an option!')

You selected the sentiment dataset. Read more about this at https://huggingface.co/datasets/cardiffnlp/tweet_eval


#### Important notes about specific datasets:

* Make sure you go to the relevant Huggingface page to read more about the [essay](https://huggingface.co/datasets/polsci/ghostbuster-essay-cleaned) and [sentiment](https://huggingface.co/datasets/cardiffnlp/tweet_eval/viewer/sentiment) datasets. Note the sentiment dataset is one subset of the larger 'tweet_eval' dataset.  
* For the *sentiment* dataset, it is challenging to get good accuracy with three classes. If you like you can remove the `neutral` class. There is a cell below that does this for you - don't change the cell above.
* For the *essay* dataset, there are differences in punctuation between classes. You should use `character_replacements = {"’": "'", '“': '"', '”': '"',}` in the `TextCleaner` component in your pipeline to make sure you are not overfitting to a quirk of the data.

This loads the dataset. 

In [34]:
if dataset_option != 'genre': # if loading from huggingface ...
    dataset = load_dataset(dataset_name, data_dir=dataset_dir)
else: # if loading the genre dataset from the provided json file
    dataset = load_dataset(dataset_type, data_files=dataset_dir)
    train_dataset = dataset['train'].filter(lambda example: example['split'] == 'train')
    test_dataset = dataset['train'].filter(lambda example: example['split'] == 'test')
    dataset = DatasetDict({
        'train': train_dataset,
        'test': test_dataset
        })

This cell will show you information on the dataset fields and the splits.

In [35]:
preview_dataset(dataset)

Here is the breakdown of the composition of labels in each data-set split.

In [36]:
# casting label column to ClassLabel if not already
cast_column_to_label(dataset, label_column)
label_names = get_label_names(dataset, label_column)

dfs = {}
for split in dataset.keys():
    dfs[split] = dataset[split].to_pandas()
    dfs[split].insert(1, 'label_name', dfs[split][label_column].apply(lambda x: dataset[split].features[label_column].int2str(x)))
    print('Labels for {}:'.format(split))
    preview_label_counts(dfs[split], label_column, label_names)

Column 'label' is already a ClassLabel.
Labels for train:


,label_name,count
label,,
0,negative,7093
1,neutral,20673
2,positive,17849


Labels for validation:


,label_name,count
label,,
0,negative,312
1,neutral,869
2,positive,819


Labels for test:


,label_name,count
label,,
0,negative,3972
1,neutral,5937
2,positive,2375


### 2.2 Configure the labels (optional)

* You can override the default labels for the data-set here to make the task more or less challenging. High accuracy does not guarantee a high grade. 
* See the assignment instructions and the dataset card or corresponding paper for explanations of the data.  
* Read the comments below and uncomment the relevant lines for your data-set if and amend the label names if needed.
* Remember, this is optional.

In [37]:
# for the movie reviews dataset (this is just for testing/demonstration) - there are 2 labels and that is it!

# for the sentiment dataset - there are 3 labels - you can make the task simpler as a binary classification problem using one of these options:
#target_labels = ['negative', 'neutral']
target_labels = ['negative', 'positive']
#target_labels = ['neutral', 'positive']

# for the essay dataset - there are 7 labels - you can make the task simpler as a binary classification problem using one of these options:
#target_labels = ['claude', 'gpt']
#target_labels = ['human', 'gpt'] 
#target_labels = ['human', 'claude']

# for the genre dataset - there are 7 labels - you can turn the task into one or more binary classification problems using options such as:
#target_labels = ['Letter', 'Notice']
#target_labels = ['Letter', 'Fiction']
#target_labels = ['Review', 'Fiction']
#target_labels = ['Notice', 'Obituary']

print(target_labels)

['negative', 'positive']


### 2.3 Prepare the train and test splits

* This cell handles the train-test split for you.
* Some of the data-sets are unbalanced. This cell will balance the training data using under-sampling.

In [38]:
target_classes = [label_names.index(name) for name in target_labels]
target_names = [label_names[i] for i in target_classes]

if train_split_name == test_split_name:
    X = dataset[train_split_name].to_pandas()
    X.insert(1, 'label_name', dfs[train_split_name][label_column].apply(lambda x: dataset[train_split_name].features[label_column].int2str(x)))
    y = np.array(dataset[train_split_name][label_column])

    mask = np.isin(y, target_classes)
    X = X.loc[mask]
    y = y[mask]

    # creating df splits with original data first  - so can look at the train data if needed
    dfs['train'], dfs['test'], y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # we're just using the text for features
    X_train = np.array(dfs['train'][text_column])
    X_test = np.array(dfs['test'][text_column])
else:
    X_train = np.array(dataset[train_split_name][text_column])
    y_train = np.array(dataset[train_split_name][label_column])
    X_test = np.array(dataset[test_split_name][text_column])
    y_test = np.array(dataset[test_split_name][label_column])

    mask = np.isin(y_train, target_classes)
    mask_test = np.isin(y_test, target_classes)

    X_train = X_train[mask]
    y_train = y_train[mask]
    X_test = X_test[mask_test]
    y_test = y_test[mask_test]

# this cell undersamples all but the minority class to balance the training data
X_train = X_train.reshape(-1, 1)
X_train, y_train = RandomUnderSampler(random_state=0).fit_resample(X_train, y_train)
X_train = X_train.reshape(-1)

preview_splits(X_train, y_train, X_test, y_test, target_classes = target_classes, target_names = target_names)

Train: 14186 samples, 2 classes


,label_name,count
0,,
0,negative,7093
2,positive,7093


Test: 1131 samples, 2 classes


,label_name,count
0,,
2,positive,819
0,negative,312


### 2.4 Preview the texts

Time to get to know your data. We will only preview the train split.

In [39]:
y_train_names = map(lambda x: label_names[x], y_train)
# Note: Version 0.2.1 corrects display of the dataframe to ensure filtering by the selected labels 
display(dfs['train'][dfs['train']['label_name'].isin(y_train_names)].sample(20))
#orginal code smaple(10), switched to 20 for more info 

,text,label_name,label
1176,Is it even possible to go to Digi on a Monday without getting paralytic,positive,2
17593,Watching the 1st series of Game of Thrones is bringing back painful memories,negative,0
22725,"""From BAMS Radio tonite... The Legend says \""""""""this BAMA defense may be the greatest of all time...\"""""""" Offense wins games, Defense wins.....""",positive,2
42460,I celebrated like Conor McGregor (into my pillow. Why frighten the neighbors?),positive,2
37302,"""Me and @user did a Popptartvision based Big Brother simulation and Kaliopi won, this proves she's queen of everything. Esma got 2nd.""",positive,2
22014,"@user relax, you can now concentrate on finishing 4th and which one of Wilshere, Carzorla or Vermalen you want to sell to City.""",positive,2
38109,@user support for 365 has been terrible. On the phone for an hour then got dropped. So far not worth the headache. May ask for refund.,negative,0
32644,wish you happy Holi - 08 March: Noel Gallagher Surprises Diners At ...: Information about Science & Tec... #asematy,positive,2
1804,Tuesday is a great day at Love Life Hot Yoga with great classes to choose from! Breathe. Move. Sweat. at 5:30 (90...,positive,2
38252,Tomorrow is the big Dark Souls race between myself and @user Starting sometime between 9-10AM EST Be there!,positive,2


## full text check 
change the selected_index to check with the original training tweet

In [40]:
selected_index = 91
# switch the index number for different comments
preview_row_text(dfs['train'], selected_index, text_column = text_column, limit=400) # change limit to see more of the text if needed

,Value
Attribute,
label_name,neutral
label,1


text:
The Hitchhiker's Guide to the Galaxy Game - 30th Anniversary Edition Need to
sign up with the BBC online.


## 3. Create a classification pipeline and train a model

Create a Sci-kit Learn pipeline to preprocess the texts and train a classification model. The pipeline components will be added in through the notebook. There are a number of pipeline components you can access through the `textplumber` package. You will have an opportunity to learn about this in labs, but documentation is [available here](https://geoffford.nz/textplumber).

To speed up preprocessing some of the pipeline components store the preprocessed data in a cache to avoid recomputing them. Run this as is - it will create an SQLite file with the name of your dataset option in the directory of the notebook. This will speed up some repeated processing (e.g. tokenization with Spacy).

In [41]:
feature_store = TextFeatureStore(f'assignment-{dataset_option}.sqlite')

The pipeline below includes a number of different components. Most are commented out on the first run of the notebook. There are lots of options for each component. You will need to look at the documentation and examples in labs to learn about these. These components can extract different kinds of features, any of which can be applied to build a model. The feature types include:

* Token features  
* Bigram features  
* Parts of speech features
* Lexicon-based features  
* Document-level statistics  
* Text embeddings


## 3.1 caution_stop_words (customised stop words list) 
This is a special word list designed to keep “non-noise” stop words, which helps the model to detect negation. Preserving sentiment-relevant negation forms would be easier for the model to recognise polarity reversal.

In [42]:
caution_stop_words = [
    i for i in stop_words
    if i not in ('not', 'no', 'never', 'nt', "n't","ever","ca")
]

## 3.2.1 Pipeline 1 model design 
Features combination: Token + POS + Lexicon

Algorithmic model: Logistic regression 

In [59]:
pipeline1 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), #change, do not keep only 2 lexicon features
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),

		 #('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	('classifier', LogisticRegression(max_iter=5000, random_state=42)) # for logistic regression - only select one classifier!
    #('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline1)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [60]:
pipeline1.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   0.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   2.5s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 3) Processing tokens, total=   2.5s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   2.0s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 3) Processing pos, total=   2.0s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   2.4s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 3) Processing lexicon, total=   2.5s
[Pipeline] .......... (step 3 of 4) Processing features, total=   6.9s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [61]:
y_predicted = pipeline1.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.633     0.830     0.718       312
    positive      0.927     0.817     0.868       819

    accuracy                          0.821      1131
   macro avg      0.780     0.823     0.793      1131
weighted avg      0.846     0.821     0.827      1131



## 3.2.2 Pipeline 2 model design 
Features combination: Token + POS + Lexicon + Embeddings

Algorithmic model: Logistic regression 

In [46]:
pipeline2 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
	 ('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	('classifier', 
     LogisticRegression(
         max_iter=5000,
         random_state=42)) # for logistic regression - only select one classifier!
    #('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline2)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [47]:
pipeline2.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.0s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   2.4s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 4) Processing tokens, total=   2.4s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   1.8s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 4) Processing pos, total=   1.8s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   2.4s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 4) Processing lexicon, total=   2.4s
[FeatureUnion] .... (step 4 of 4) Processing embeddings, total=   1.3s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [48]:
y_predicted = pipeline2.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.690     0.856     0.764       312
    positive      0.940     0.853     0.894       819

    accuracy                          0.854      1131
   macro avg      0.815     0.855     0.829      1131
weighted avg      0.871     0.854     0.858      1131



## 3.2.3 Pipeline 3 model design 
Features combination: Token + POS + Lexicon

Algorithmic model: Decision tree

In [49]:
pipeline3 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), #change, do not keep only 2 lexicon features
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),

		 #('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	#('classifier', LogisticRegression(max_iter=5000, random_state=42)) # for logistic regression - only select one classifier!
    ('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline3)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [50]:
pipeline3.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.0s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   2.4s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 3) Processing tokens, total=   2.4s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   1.7s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 3) Processing pos, total=   1.8s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   2.5s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 3) Processing lexicon, total=   2.6s
[Pipeline] .......... (step 3 of 4) Processing features, total=   6.8s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [51]:
y_predicted = pipeline3.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.484     0.756     0.590       312
    positive      0.882     0.692     0.776       819

    accuracy                          0.710      1131
   macro avg      0.683     0.724     0.683      1131
weighted avg      0.772     0.710     0.724      1131



## 3.2.4 Pipeline 4 model design 
Features combination: Token + POS + Lexicon + Embeddings

Algorithmic model: Decision tree

In [52]:
pipeline4 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
	 ('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	#('classifier', LogisticRegression(max_iter=5000,random_state=42)) # for logistic regression - only select one classifier!
    ('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline4)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [53]:
pipeline4.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   0.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   2.5s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 4) Processing tokens, total=   2.5s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   1.8s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 4) Processing pos, total=   1.8s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   2.5s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 4) Processing lexicon, total=   2.6s
[FeatureUnion] .... (step 4 of 4) Processing embeddings, total=   1.2s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>,...
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f07c9e6a840>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f07a87e0e30>))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [54]:
y_predicted = pipeline4.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.526     0.712     0.605       312
    positive      0.873     0.756     0.810       819

    accuracy                          0.744      1131
   macro avg      0.700     0.734     0.708      1131
weighted avg      0.777     0.744     0.754      1131



## 4. Evaluate your model and investigate model predictions

You already have some metrics in the cell above. Below is some additional reporting to help you understand your model.

### 4.1 Classifier-specific features

If you are using a Decision Tree classifier in your pipeline, this will plot it ...

In [55]:
if pipeline1.named_steps['classifier'].__class__.__name__ == 'LogisticRegression':
	plot_logistic_regression_features_from_pipeline(pipeline1, target_classes, target_names, top_n=20, classifier_step_name = 'classifier', features_step_name = 'features')

,Feature,Log Odds (Logit),Odds Ratio
706,lexicon__negative,-0.955462,0.384634
705,lexicon__positive,0.767853,2.155134
0,tokens__!,0.532720,1.703560
571,tokens__not,-0.463040,0.629367
320,tokens__Saudi Arabia,-0.440602,0.643649
377,tokens__Yakub,-0.435126,0.647183
694,tokens__worst,-0.364160,0.694780
555,tokens__n't,-0.361591,0.696567
112,tokens__Arabia,0.353184,1.423593
148,tokens__Cream Day,0.347849,1.416018


### 4.2 Investigate correct and incorrect predictions

To see the predictions of your model run this cell. The output can be quite long depending on the dataset and the number of misclassifications. The Pandas `max_rows` is configured at the top of the cell to restrict the length of output. You can adjust this as required. This is reset back to the Pandas default at the end of the cell.

In [63]:
# adjust max rows
pd.set_option('display.max_rows', 10) # show all rows

# creating dataframe from y_predicted, y_test and the text
predictions_df = pd.DataFrame(data = {'true': y_test, 'predicted': y_predicted})
y_predicted_probs = pipeline1.predict_proba(X_test)
y_predicted_probs = np.round(y_predicted_probs, 3)
# Note: Version 0.2.2 changed the following line to ensure probability labels are correct regardless of the order of target classes
columns = [f'{label_names[c]}_prob' for c in pipeline1.named_steps['classifier'].classes_ if c in target_classes]
predictions_df['predicted'] = predictions_df['predicted'].apply(lambda x: label_names[x])
predictions_df['true'] = predictions_df['true'].apply(lambda x: label_names[x])
predictions_df['correct'] = predictions_df['true'] == predictions_df['predicted']
predictions_df['text'] = X_test
predictions_df = pd.concat([predictions_df, pd.DataFrame(y_predicted_probs, columns=columns)], axis=1)

# output a preview of docs for each cell of confusion matrix ...
for true_target, target_name in enumerate(target_names):
    for predicted_target, target_name in enumerate(target_names):
        if true_target == predicted_target:
            print(f'\nCORRECTLY CLASSIFIED: {target_names[true_target]}')
        else:
            print(f'\n{target_names[true_target]} INCORRECTLY CLASSIFIED as: {target_names[predicted_target]}')
        print('=================================================================')

        display(predictions_df[(predictions_df['true'] == target_names[true_target]) & (predictions_df['predicted'] == target_names[predicted_target])])

pd.set_option('display.max_rows', 60) # setting back to the default


CORRECTLY CLASSIFIED: negative


,true,predicted,correct,text,negative_prob,positive_prob
1,negative,negative,True,When girls become bandwagon fans of the Packers because of Harry. Do y'all even know who Aaron Rodgers is? Or what a 1st down is?,0.745,0.255
3,negative,negative,True,Omg this show is so predictable even for the 3rd ep. Rui En\u2019s ex boyfriend was framed for murder probably\u002c by the rich guy.,0.866,0.134
5,negative,negative,True,"@user so the thing next Thursday isn't free, you'd have to pay $15 to get in since you don't go to UMBC :/ and it ends at 11:30""",0.966,0.034
11,negative,negative,True,The sad part about this is tomorrow Nicki will be the angry black woman who went after poor white girl Miley,0.997,0.003
13,negative,negative,True,@user I haven't been able to watch TVD live these days due to Football. Every Thurs there is high school FB going on at 8pm. Like WTF!,0.620,0.380
...,...,...,...,...,...,...
1116,negative,negative,True,It is reality that ISIS are on the march in Turkey and Erdogan can't wait to receive them with open arms,0.881,0.119
1122,negative,negative,True,Disappointed the Knicks vs Nets game got canceled tonight\u002c but I\u2019m even more hyped for Knicks vs Heat on Friday!,0.784,0.216
1126,negative,negative,True,"@user @user Islam is an Abrahamic faith, Andrew. It may make you feel a little uneasy but it's the same God you worship. Sorry.""",0.803,0.197
1127,negative,negative,True,kingpin Saudi Arabia posted a record $98 billion budget deficit in 2015 due to the sharp fall in oil prices finance ministry said on Monday,0.979,0.021



negative INCORRECTLY CLASSIFIED as: positive


,true,predicted,correct,text,negative_prob,positive_prob
7,negative,positive,False,F*** the hurricane party this Tues santospartyhaus w/ @user @user @user @ Santos Party House,0.115,0.885
52,negative,positive,False,"""Judging by the traffic and complaining, I think I might be best setting for Foo Fighters' Milton Keynes gig tomorrow right now""",0.068,0.932
67,negative,positive,False,"""Galaxy S II can be wiped by just clicking a link, other devices may be vulnerable #Galaxy #Samsung #HTML #Vulnerability""",0.420,0.580
128,negative,positive,False,just bought my 1st Heineken beer in Las Vegas. ps I\u2019ve lived here for 5 yrs ~what took me so long!,0.152,0.848
132,negative,positive,False,Monday Night Raw is on on Monday nights from 7pm-10pm. The 3rd season of Teen Mom 2 is on a new night. Mondays. At 9pm. Oh the dilemma......,0.130,0.870
...,...,...,...,...,...,...
946,negative,positive,False,Take note: Putting stylus back in Galaxy Note 5 may break the phone Read more at:,0.376,0.624
1027,negative,positive,False,If you're seeing the weekend instead of Paul McCartney at lolla Friday I am judging the fuck out of you.,0.119,0.881
1038,negative,positive,False,"""\""""""""@nodoubt: Tune into @user tomorrow for a special @user #PushAndShove News segment during the 7AM & 9AM hours!\"""""""" NOOOOOOOOO""",0.243,0.757
1084,negative,positive,False,David Cameron's statement on camera on Thursday 03 September 2015: he will take in 'more' of the refugees: was he speaking TO TV Cameras?,0.434,0.566



positive INCORRECTLY CLASSIFIED as: negative


,true,predicted,correct,text,negative_prob,positive_prob
2,positive,negative,False,#US 1st Lady Michelle Obama speaking at the 2015 Beating the Odds Summit to over 130 college-bound students at the pentagon office. &gt;&gt;,0.975,0.025
17,positive,negative,False,tomorrow I've to wake up early so Zayn's erformance on VMA better be true otherwise u'll regret for playing with my emotions and sleep,0.600,0.400
19,positive,negative,False,Nicki did that for white media Idgaf . Nicki may act like she don't give af but she cares what the media thinks,0.957,0.043
22,positive,negative,False,@user You may check with Amazon for Kindle as it's screen is glare free and can be read in Sunlight too.,0.594,0.406
25,positive,negative,False,I love how Thanksgiving break had hella bangers and we're in the 2nd week of winter break and have failed to have one yet,0.555,0.445
...,...,...,...,...,...,...
1093,positive,negative,False,YOU MUST READ! Karina Smirnoff Wows in a Low-Cut Yellow Dress at the ESPY Awards (PHOTOS),0.770,0.230
1096,positive,negative,False,"""The sun shall not smite I by day, nor the moon by night"" Bob Marley &amp; the Wailers NIGHT SHIFT *Live in NYC 4/30/76*",0.522,0.478
1098,positive,negative,False,When I wake up tomorrow I'll be in a different country. Whoa! I didn't run into a David Beckham at the airport. That's a bummer.,0.741,0.259
1109,positive,negative,False,This is going to be a fun 5 years. The makings of the next Prime Minister... George Osborne....,0.531,0.469



CORRECTLY CLASSIFIED: positive


,true,predicted,correct,text,negative_prob,positive_prob
0,positive,positive,True,"""National hot dog day, national tequila day, then national dance day... Sounds like a Friday night.""",0.315,0.685
4,positive,positive,True,"""What a round by Paul Dunne, good luck tomorrow and I hope you win the Open.""",0.012,0.988
6,positive,positive,True,Can't wait for tomorrow at 9 pm! Chelsea v crystal palace! Hope the blues will win!,0.000,1.000
8,positive,positive,True,"""Thank you @user for the message. I'm very proud to be a Liverpudlian, may i get your follback? #LiverpudlianLoyalitasTanpaBatas #YNWA""",0.086,0.914
9,positive,positive,True,Tom Brady is locked for Thursday. Let the season begin! #RepeatSeason,0.361,0.639
...,...,...,...,...,...,...
1121,positive,positive,True,This Saturday &amp; Sunday come join us the @user at the Pomona Fairplex! Your ticket can WIN you a Brand New Car!,0.007,0.993
1123,positive,positive,True,PM ready for reply on coal blocks: Congress: New Delhi\u002c Aug 22 (IANS) With the Bharatiya Janata Party (BJP)...,0.061,0.939
1124,positive,positive,True,"""\""""""""@_eryflores: March 16 Luke Bryan is gonna at the Houston Rodeo. I HAVE to go\u002c Its a MUST!\""""""""""",0.494,0.506
1129,positive,positive,True,Beautiful Bouquet with our Beautiful Bentley #bride #groom #wedding #wednesday #weddingcars #love #Repost...,0.010,0.990


In [67]:
selected_index = 25

preview_row_text(predictions_df, selected_index, text_column = text_column, limit=400) # change limit to see more of the text if needed

,Value
Attribute,
true,positive
predicted,negative
correct,False
negative_prob,0.555
positive_prob,0.445


text:
I love how Thanksgiving break had hella bangers and we're in the 2nd week of
winter break and have failed to have one yet


### 4.3 Run inference on new (or old) data

You can also run inference on new data (or any of the texts from training/validation) by changing the contents of the `texts` list below. This outputs a prediction, the probabilities of each class and the features present within the text that are used by the model to make its predictions. The numbers for each feature are the input to the final step of the pipeline. They may be scaled or transformed depending on the pipeline components you've chosen.

## 4.3.1 Explore negative INCORRECTLY CLASSIFIED as: positive
1 tweet example selected to present in the report. 

In [65]:
texts = ['''
Judging by the traffic and complaining, I think I might be best setting for Foo
Fighters' Milton Keynes gig tomorrow right now
''',
]

y_inference = pipeline1.predict(texts)

preprocessor = Pipeline(pipeline1.steps[:-1])
feature_names = preprocessor.named_steps['features'].get_feature_names_out()

for i, text in enumerate(texts):
	print(f"Text {i}: {text}")
	
	print(f"\tPredicted class: {label_names[y_inference[i]]}")
	print()

	y_inference_proba = pipeline1.predict_proba([text])
	
	# Note: Version 0.2.2 changed the following lines to ensure probability labels are correct regardless of the order of target classes
	for idx, prob in enumerate(y_inference_proba[0]):
		c = pipeline1.named_steps['classifier'].classes_[idx]
		if c in target_classes:
			print(f"\tProbability of class {label_names[c]}: {prob:.2f}")
	# End change for 0.2.2

	print()
	print("\tFeatures:")

	embeddings = 0
    
	frequencies = preprocessor.transform([text])
	if not isinstance(frequencies, np.ndarray):
		frequencies = frequencies.toarray()
	frequencies = frequencies[0].T
    
	for j, freq in enumerate(frequencies):
		if feature_names[j].startswith('embeddings_'):
			embeddings += 1
		elif freq > 0:
			print(f"\t{feature_names[j]}: {freq:.2f}")
	if embeddings > 0:
		print(f"\tFeatures also include {embeddings} embedding dimensions")

	print()


Text 0: 
Judging by the traffic and complaining, I think I might be best setting for Foo
Fighters' Milton Keynes gig tomorrow right now

	Predicted class: positive

	Probability of class negative: 0.05
	Probability of class positive: 0.95

	Features:
	tokens__': 3.97
	tokens__,: 1.28
	tokens__, I: 6.11
	tokens__Fighters: 12.85
	tokens__Foo: 12.66
	tokens__Foo Fighters: 12.85
	tokens__I: 2.80
	tokens__I think: 8.49
	tokens__best: 5.32
	tokens__right: 7.27
	tokens__think: 5.39
	tokens__tomorrow: 2.22
	pos__AUX: 1.60
	pos__PART: 1.26
	pos__PROPN: 1.76
	pos__VERB: 2.18
	lexicon__positive: 1.13



## 4.3.2 Explore positive INCORRECTLY CLASSIFIED as: negative 
1 tweet example selected to present in the report. 

In [68]:
texts = ['''
I love how Thanksgiving break had hella bangers and we're in the 2nd week of
winter break and have failed to have one yet
''',
]

y_inference = pipeline1.predict(texts)

preprocessor = Pipeline(pipeline1.steps[:-1])
feature_names = preprocessor.named_steps['features'].get_feature_names_out()

for i, text in enumerate(texts):
	print(f"Text {i}: {text}")
	
	print(f"\tPredicted class: {label_names[y_inference[i]]}")
	print()

	y_inference_proba = pipeline1.predict_proba([text])
	
	# Note: Version 0.2.2 changed the following lines to ensure probability labels are correct regardless of the order of target classes
	for idx, prob in enumerate(y_inference_proba[0]):
		c = pipeline1.named_steps['classifier'].classes_[idx]
		if c in target_classes:
			print(f"\tProbability of class {label_names[c]}: {prob:.2f}")
	# End change for 0.2.2

	print()
	print("\tFeatures:")

	embeddings = 0
    
	frequencies = preprocessor.transform([text])
	if not isinstance(frequencies, np.ndarray):
		frequencies = frequencies.toarray()
	frequencies = frequencies[0].T
    
	for j, freq in enumerate(frequencies):
		if feature_names[j].startswith('embeddings_'):
			embeddings += 1
		elif freq > 0:
			print(f"\t{feature_names[j]}: {freq:.2f}")
	if embeddings > 0:
		print(f"\tFeatures also include {embeddings} embedding dimensions")

	print()


Text 0: 
I love how Thanksgiving break had hella bangers and we're in the 2nd week of
winter break and have failed to have one yet

	Predicted class: negative

	Probability of class negative: 0.55
	Probability of class positive: 0.45

	Features:
	tokens__2nd: 3.90
	tokens__I: 1.30
	tokens__I love: 10.57
	tokens__break: 25.22
	tokens__love: 5.41
	tokens__one: 4.27
	tokens__week: 8.98
	tokens__yet: 9.31
	pos__AUX: 3.21
	pos__PART: 1.26
	pos__PROPN: 0.44
	pos__SCONJ: 1.85
	pos__VERB: 1.45
	lexicon__positive: 1.13
	lexicon__negative: 1.66

